In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

# Googleドライブの空き容量を確認（G4用の超大型モデルは合計40〜80GBになるため）
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
free_gb = free / (1024**3)
print(f"💾 Googleドライブ空き容量: {free_gb:.1f} GB")

# 空き容量が60GB以上あればドライブに永続保存、足りなければColab本体の高速SSDに保存
if free_gb >= 60:
    os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/ollama_models'
    print("✅ モデルはGoogleドライブ (/content/drive/MyDrive/ollama_models) に永続保存されます。")
else:
    os.environ['OLLAMA_MODELS'] = '/root/.ollama/models'
    print("⚠️ ドライブの空きが60GB未満のため、容量不足エラーを防ぐためColab本体の高速SSDに保存します。")


In [ ]:
import os
import subprocess
import time
import shutil
import re

# 破損ファイルや過去のゴミをクリーンアップ
for path in ["/usr/local/bin/ollama", "/usr/bin/ollama", "/content/cloudflared.deb"]:
    if os.path.exists(path):
        try:
            if os.path.isdir(path):
                shutil.rmtree(path)
            else:
                os.remove(path)
        except Exception:
            pass

def download_file(url, filepath):
    try:
        subprocess.run(["curl", "-L", "-o", filepath, url], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

# 0. 依存パッケージ (zstd + G4 Blackwell GPU自動検出用pciutils) をインストール
print("📦 依存パッケージ (zstd, pciutils) をインストール中...")
subprocess.run(["apt-get", "update", "-y"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "zstd", "pciutils"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 1. Ollama のインストール (G4 / RTX PRO 6000 Blackwell アーキテクチャを自動認識)
print("📦 Ollama をインストール中...")
!curl -fsSL https://ollama.com/install.sh | sh

# 2. cloudflared のインストール
print("📦 Cloudflare Tunnel を準備中...")
cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
success_cf = download_file(cf_url, "/content/cloudflared.deb")

if success_cf and os.path.exists("/content/cloudflared.deb") and os.path.getsize("/content/cloudflared.deb") > 5000000:
    !dpkg -i /content/cloudflared.deb > /dev/null 2>&1
    print("✅ Cloudflare Tunnel の準備完了")
else:
    raise RuntimeError("❌ Cloudflare Tunnel のダウンロードに失敗しました。")

# 3. G4 GPU (RTX PRO 6000 Blackwell / 96GB VRAM) 最適化パラメーターで Ollama を起動
print("🚀 Ollama サーバーを G4 GPU (96GB VRAM) 最適化パラメーターで起動中...")
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
os.environ['OLLAMA_CONTEXT_LENGTH'] = '131072'    # ★G4の96GB超大容量VRAMを活かして128Kコンテキストに拡張
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'        # ★Blackwell世代のFlash Attentionを有効化（長文の超高速化＆省メモリ化）
os.environ['OLLAMA_KV_CACHE_TYPE'] = 'q8_0'       # ★KVキャッシュを8bit量子化（128K長文でもVRAM消費を大幅削減）
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '2'      # ★96GBあるので30B級モデル2つ（例: Qwen + Gemma 4）を同時にVRAM常駐可能
os.environ['OLLAMA_NUM_PARALLEL'] = '2'           # ★並列リクエスト処理数を2に強化
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'            # ★モデルをVRAMに永続常駐（モデル切り替えの待ち時間ゼロ）

subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

# 4. G4 GPU (96GB VRAM) おすすめAIモデル（80B超大型＆Google最上位モデル・ツール呼び出し対応）
MODELS = [
    "qwen3-coder-next",   # ★G4の真骨頂(80B MoE / 実効3B): 96GB VRAMならではの超大型エージェント特化モデル（爆速＆超高知能）
    "gemma4:31b",         # ★Google最高峰(31B Dense): Google Gemma 4 フラッグシップ（高度な推論・コード・マルチモーダル・ツール対応）
    "gemma4:26b",         # ★Google超高速版(26B MoE): Gemma 4 MoEモデル（実効4Bで瞬時に応答・ツール呼び出し対応）
    "qwen3-coder:30b",    # ★定番エージェント特化(30B MoE): 安定感抜群の高速コーディングモデル
    # "llama3.3:70b",      # （任意）Metaの70B超大型モデル（約43GB・G4の96GBなら余裕で動作）
    # "qwen3:32b",         # （任意）32B Denseフルモデル（複雑な論理推論向け）
]

for model in MODELS:
    print(f"\n📥 モデル ({model}) を準備中...")
    !ollama pull {model}

# 5. Cloudflare Tunnel を起動してURLを発行
print("\n🌐 Cloudflare Tunnel を起動中...")
log_path = "/content/cloudflared.log"
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434",
     "--http-host-header", "localhost:11434"],
    stdout=open(log_path, "w"), stderr=subprocess.STDOUT
)

url = None
for _ in range(60):
    time.sleep(1)
    if os.path.exists(log_path):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
        if m:
            url = m.group(0)
            break

if url:
    print("\n" + "="*65)
    print("🎉 G4 GPU (96GB VRAM) 準備完了！以下のURLを Zoo Code / Cline の Base URL に貼ってください：")
    print(f"👉 {url}")
    print("="*65)
    print("📌 使えるモデルID一覧（Zoo CodeのModel欄に入力）：")
    for m in MODELS:
        print(f"   • {m}")
    print("\n⚠️ このノートブックを閉じるとURLは無効になります。")
else:
    print("❌ URLを取得できませんでした。/content/cloudflared.log を確認してください。")
